In [8]:
# ==============================================================================
# 1: 导入与全局配置
# ==============================================================================
import pandas as pd
import numpy as np
import os
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import GridSearchCV, train_test_split, StratifiedKFold
from sklearn.metrics import make_scorer
def gfc_each(y_true, y_pred):
    n_t, n_p = np.linalg.norm(y_true, axis=1), np.linalg.norm(y_pred, axis=1)
    dot = np.sum(y_true * y_pred, axis=1)
    return np.where(n_t * n_p == 0, 0, dot / (n_t * n_p))
def gfc_mean(y_true, y_pred):
    return np.mean(gfc_each(y_true, y_pred))
scorer = make_scorer(gfc_mean, greater_is_better=True)
# ==============================================================================
# 2: 数据读取
# ==============================================================================
file_path = r"D:\project\数据整理.xlsx"
output_file = r"D:\project\RF预测结果_GFC评分.xlsx" 
sheet_as = "AS7343去除11条的异常数据"
sheet_ev = "EVERFINE去除11条的异常数据"
df_as = pd.read_excel(file_path, sheet_name=sheet_as)
df_ev = pd.read_excel(file_path, sheet_name=sheet_ev)
ids = df_as.iloc[:, 0].values
X = df_as.iloc[:, 1:].values
Y = df_ev.iloc[:, 1:].values
# ==============================================================================
# 3: 初步建模
# ==============================================================================
indices = np.arange(len(ids))
stratify_labels = (ids > 343).astype(int)
X_tr, X_te, Y_tr, Y_te, idx_tr, idx_te = train_test_split(
    X, Y, indices, 
    test_size=0.2, 
    random_state=42,
    stratify=stratify_labels 
)
model_base = MultiOutputRegressor(RandomForestRegressor(random_state=42))
model_base.fit(X_tr, Y_tr)
score_base = scorer(model_base, X_te, Y_te)
raw_params = model_base.estimator.get_params()
print("基准模型参数组合:", {k: raw_params[k] for k in [
    'n_estimators',
    'max_depth', 
    'min_samples_leaf', 
    'min_samples_split', 
    'max_features']})
print(f"基准模型测试集平均 GFC: {score_base:.4f}")
# ==============================================================================
# 4: 参数寻优
# ==============================================================================
param_grid = { 
    'estimator__n_estimators': [100],     
    'estimator__max_depth': [10, 20, None],     
    'estimator__max_features': [1.0],
    'estimator__min_samples_split': [2],     
    'estimator__min_samples_leaf': [1],  
}
grid_search = GridSearchCV(
    estimator=MultiOutputRegressor(RandomForestRegressor(random_state=42)), 
    param_grid=param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(X, stratify_labels),
    scoring=scorer,           
    n_jobs=-1,                      
)
grid_search.fit(X, Y)
best_params = grid_search.best_params_
print("\n最优参数组合:")
print(best_params)
print(f"\n五折交叉验证最优平均 GFC 评分: {grid_search.best_score_:.4f}")
# ==============================================================================
# 5: 最优模型训练与评估
# ==============================================================================
clean_params = {k.replace('estimator__', ''): v for k, v in best_params.items()}
model_final = MultiOutputRegressor(RandomForestRegressor(random_state=42, **clean_params))
model_final.fit(X_tr, Y_tr)
score_best = scorer(model_final, X_te, Y_te)
print(f"最优模型测试集平均 GFC: {score_best:.4f}")
# ==============================================================================
# 6: 结果输出
# ==============================================================================
pred_all = model_final.predict(X)
gfc_all = gfc_each(Y, pred_all) 
data_type_labels = ['训练集'] * len(ids)
for idx in idx_te:
    data_type_labels[idx] = '测试集'
res_df = pd.DataFrame(columns=df_ev.columns)
res_df.iloc[:, 0] = ids
res_df.iloc[:, 1:] = pred_all
res_df['GFC'] = gfc_all
res_df['数据集类型'] = data_type_labels 
os.makedirs(os.path.dirname(output_file), exist_ok=True)
res_df.to_excel(output_file, index=False)

基准模型参数组合: {'n_estimators': 100, 'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'max_features': 1.0}
基准模型测试集平均 GFC: 0.9889

最优参数组合:
{'estimator__max_depth': 20, 'estimator__max_features': 1.0, 'estimator__min_samples_leaf': 1, 'estimator__min_samples_split': 2, 'estimator__n_estimators': 100}

五折交叉验证最优平均 GFC 评分: 0.9843
最优模型测试集平均 GFC: 0.9890
